# HEADS Training Notebook (Saves Models)

This notebook trains the HEADS pipeline and saves model artifacts into `models/` so the dashboard shows **Trained models ✅**.

Saved files:
- `models/transformer_ae.pt`
- `models/graphsage.pt`
- `models/xgb.json` *(if labels exist)*
- `models/tabular_featurizer.joblib` *(if labels exist)*
- `data/processed/scored_events.csv`


In [ ]:
import pandas as pd, numpy as np, re
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import joblib
from xgboost import XGBClassifier


In [ ]:
DATA_PATH = Path('data/raw/cybersecurity.csv')
MODEL_DIR = Path('models'); MODEL_DIR.mkdir(exist_ok=True)
OUT_DIR = Path('data/processed'); OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN = 20
AE_EPOCHS = 5
GNN_EPOCHS = 5
DEVICE


In [ ]:
REQUIRED_COLS = ['timestamp','src_ip','dst_ip','src_port','dst_port','protocol','bytes_sent','bytes_received','user_agent','url','is_internal_traffic']
def load_csv(path: Path):
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    for c in ['src_port','dst_port','bytes_sent','bytes_received']:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)
    df['protocol'] = df['protocol'].astype(str).fillna('UNK')
    df['user_agent'] = df['user_agent'].astype(str).fillna('UNK')
    df['url'] = df['url'].astype(str).replace('nan','').fillna('')
    df['is_internal_traffic'] = df['is_internal_traffic'].astype(bool)
    if 'label' in df.columns:
        df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
    if 'attack_type' in df.columns:
        df['attack_type'] = df['attack_type'].astype(str).fillna('benign')
    return df.sort_values(['src_ip','timestamp'], kind='mergesort').reset_index(drop=True)

def extract_url_host(url: str) -> str:
    m = re.match(r'^https?://([^/]+)/?', str(url))
    return m.group(1).lower() if m else ''

def add_feats(df):
    df = df.copy()
    df['hour'] = df['timestamp'].dt.hour.fillna(0).astype(int)
    df['dow'] = df['timestamp'].dt.dayofweek.fillna(0).astype(int)
    df['url_host'] = df['url'].apply(extract_url_host)
    df['bytes_total'] = (df['bytes_sent'] + df['bytes_received']).astype(int)
    df['is_web'] = df['dst_port'].isin([80,443]).astype(int)
    df['is_internal_traffic'] = df['is_internal_traffic'].astype(int)
    return df

df = add_feats(load_csv(DATA_PATH))
df.head()


## Train Transformer Autoencoder and save `models/transformer_ae.pt`


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X): self.X = X.astype(np.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return torch.from_numpy(self.X[i])

class TransformerAE(nn.Module):
    def __init__(self, feat_dim, d_model=64, nhead=4, layers=2, dropout=0.1):
        super().__init__()
        self.proj_in = nn.Linear(feat_dim, d_model)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        dec = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerEncoder(dec, num_layers=max(1,layers))
        self.proj_out = nn.Linear(d_model, feat_dim)
    def forward(self, x):
        z = self.proj_in(x)
        z = self.encoder(z)
        z = self.decoder(z)
        return self.proj_out(z)

def build_sequences(df, cols, seq_len=20):
    counts = df.groupby('src_ip').size()
    eff = max(3, min(seq_len, int(counts.max())))
    X_list=[]; idx=[]
    for _, g in df.groupby('src_ip', sort=False):
        g=g.sort_values('timestamp')
        if len(g) < eff: continue
        X = g[cols].to_numpy(float)
        for i in range(eff-1, len(g)):
            X_list.append(X[i-eff+1:i+1]); idx.append(g.index[i])
    if len(X_list) < 50:
        g = df.sort_values('timestamp')
        X = g[cols].to_numpy(float)
        X_list=[]; idx=[]
        for i in range(eff-1, len(g)):
            X_list.append(X[i-eff+1:i+1]); idx.append(g.index[i])
    return np.stack(X_list), np.array(idx), eff

ae_cols = ['src_port','dst_port','bytes_sent','bytes_received','bytes_total','hour','dow','is_web','is_internal_traffic']
X_seq, idx_last, eff = build_sequences(df, ae_cols, SEQ_LEN)
ae = TransformerAE(len(ae_cols)).to(DEVICE)
dl = DataLoader(SeqDataset(X_seq), batch_size=256, shuffle=True)
opt = torch.optim.AdamW(ae.parameters(), lr=3e-4)
loss_fn = nn.MSELoss()
for ep in range(AE_EPOCHS):
    ae.train(); tot=0.0
    for x in dl:
        x=x.to(DEVICE)
        xh=ae(x)
        loss=loss_fn(xh,x)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += float(loss.item())*x.size(0)
    print('AE epoch', ep+1, 'loss', tot/len(dl.dataset))

with torch.no_grad():
    ae.eval()
    x = torch.tensor(X_seq, dtype=torch.float32, device=DEVICE)
    xh = ae(x)
    scores = torch.mean((xh-x)**2, dim=(1,2)).detach().cpu().numpy()
df['temporal_score'] = 0.0
df.loc[idx_last, 'temporal_score'] = scores
torch.save(ae.state_dict(), MODEL_DIR/'transformer_ae.pt')
MODEL_DIR/'transformer_ae.pt'


## Train GraphSAGE and save `models/graphsage.pt`


In [ ]:
def build_graph(df):
    node_map={}; node_types=[]; edges=[]
    def get_node(t,key):
        k=(t,str(key))
        if k in node_map: return node_map[k]
        nid=len(node_map); node_map[k]=nid; node_types.append(t); return nid
    for _,r in df.iterrows():
        a=get_node('actor', r['src_ip']); b=get_node('resource', r['dst_ip']); edges.append((a,b))
        if r.get('url_host',''):
            edges.append((a, get_node('host', r['url_host'])))
        edges.append((a, get_node('proto', r['protocol'])))
    edge_index = np.array(edges, dtype=np.int64).T
    types_sorted = sorted(set(node_types)); t2i={t:i for i,t in enumerate(types_sorted)}
    type_ids = np.array([t2i[t] for t in node_types], dtype=np.float32)
    deg = np.zeros(len(node_map), dtype=np.float32)
    for u,v in edges: deg[u]+=1; deg[v]+=1
    x = np.stack([type_ids, np.log1p(deg)], axis=1)
    data = Data(x=torch.tensor(x, dtype=torch.float32), edge_index=torch.tensor(edge_index, dtype=torch.long), num_nodes=len(node_map))
    return data, node_map

class SAGE(nn.Module):
    def __init__(self, in_dim=2, hidden=64, layers=2):
        super().__init__()
        self.convs=nn.ModuleList([SAGEConv(in_dim, hidden)])
        for _ in range(layers-1): self.convs.append(SAGEConv(hidden, hidden))
    def forward(self, x, edge_index):
        for c in self.convs:
            x = torch.relu(c(x, edge_index))
        return x

def neg_sample(n, k):
    return torch.stack([torch.randint(0,n,(k,),device=DEVICE), torch.randint(0,n,(k,),device=DEVICE)], dim=0)

data, node_map = build_graph(df)
data = data.to(DEVICE)
gnn = SAGE(in_dim=data.x.size(1)).to(DEVICE)
opt = torch.optim.Adam(gnn.parameters(), lr=1e-3)
E = data.edge_index.size(1)
for ep in range(GNN_EPOCHS):
    gnn.train()
    z = gnn(data.x, data.edge_index)
    s,t = data.edge_index
    pos = (z[s]*z[t]).sum(dim=1)
    ne = neg_sample(data.num_nodes, E)
    ns,nt = ne
    neg = (z[ns]*z[nt]).sum(dim=1)
    loss = -torch.mean(torch.log(torch.sigmoid(pos)+1e-9)) - torch.mean(torch.log(torch.sigmoid(-neg)+1e-9))
    opt.zero_grad(); loss.backward(); opt.step()
    print('GNN epoch', ep+1, 'loss', float(loss))

with torch.no_grad():
    gnn.eval()
    z = gnn(data.x, data.edge_index)
    pairs=[]
    for _,r in df.iterrows():
        pairs.append((node_map[('actor', str(r['src_ip']))], node_map[('resource', str(r['dst_ip']))]))
    ei = torch.tensor(np.array(pairs, dtype=np.int64).T, device=DEVICE)
    s,t = ei
    score = (z[s]*z[t]).sum(dim=1)
    rel = (-torch.log(torch.sigmoid(score)+1e-9)).detach().cpu().numpy()
df['relational_score'] = rel
torch.save(gnn.state_dict(), MODEL_DIR/'graphsage.pt')
MODEL_DIR/'graphsage.pt'


## Train XGBoost (SMOTE) and save `models/xgb.json` + featurizer


In [ ]:
class Featurizer:
    def __init__(self, text_dim=64):
        self.ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        self.hv = HashingVectorizer(n_features=text_dim, alternate_sign=False, norm=None)
    def fit(self, df):
        self.ohe.fit(df[['protocol','url_host','is_internal_traffic']].astype(str))
        return self
    def transform(self, df):
        num = df[['src_port','dst_port','bytes_sent','bytes_received','bytes_total','hour','dow','is_web']].to_numpy(float)
        cat = self.ohe.transform(df[['protocol','url_host','is_internal_traffic']].astype(str))
        txt = (df['url'].fillna('') + ' ' + df['user_agent'].fillna('')).astype(str).tolist()
        tmat = self.hv.transform(txt).toarray().astype(float)
        return np.hstack([num, cat, tmat])

if 'label' in df.columns:
    feat = Featurizer().fit(df)
    X = np.hstack([feat.transform(df), df[['temporal_score','relational_score']].to_numpy(float)])
    y = df['label'].astype(int).to_numpy()
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    k = min(5, max(1, int((y_tr==1).sum()-1)))
    sm = SMOTE(random_state=42, k_neighbors=k)
    X_tr2, y_tr2 = sm.fit_resample(X_tr, y_tr)
    xgb = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
                        objective='binary:logistic', eval_metric='auc', n_jobs=-1, random_state=42)
    xgb.fit(X_tr2, y_tr2)
    p = xgb.predict_proba(X_te)[:,1]
    pr = (p>=0.5).astype(int)
    print('ROC-AUC', roc_auc_score(y_te, p))
    print('PR-AUC', average_precision_score(y_te, p))
    print(classification_report(y_te, pr, zero_division=0))
    df['xgb_proba'] = xgb.predict_proba(X)[:,1]
    df['prediction'] = (df['xgb_proba']>=0.5).astype(int)
    xgb.save_model(str(MODEL_DIR/'xgb.json'))
    joblib.dump(feat, MODEL_DIR/'tabular_featurizer.joblib')
    (MODEL_DIR/'xgb.json', MODEL_DIR/'tabular_featurizer.joblib')
else:
    print('No label column: XGBoost will not be trained. AE and GNN models are saved.')


In [ ]:
out_csv = OUT_DIR/'scored_events.csv'
df.to_csv(out_csv, index=False)
out_csv
